# SUPPORT generalized training with validation

Train one model from every TIFF under `in_path`, then measure the same self-supervised loss and overwrite the latest denoised output for every TIFF under `test_dir` after each epoch.

## Initialization

In [ ]:
"""Change these for your setup."""
path_SUPPORT = "/data/garfinkeljb/SUPPORT"  # no ending slash

In [ ]:
import gc
import json
import os
import sys
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.chdir(f"{path_SUPPORT}/notebooks/")
sys.path.insert(0, str(Path("./").resolve().parent.parent))

import torch

torch.cuda.empty_cache()
from SUPPORT.utils.train_collection import TrainingClass
from SUPPORT.utils.validation import ValidationTracker

In [ ]:
in_path = "/path/to/training_data"  # training directory; TIFFs are found recursively
test_dir = "/path/to/validation_data"  # held-out TIFF file or directory; directories are searched recursively
dimensionality = "2+1D"  # options: "2+1D", "3D", or "3+1D"

exp_name = str(Path(in_path).stem)
out_dir = None
validation_output_dir = None  # defaults to <training output>/validation

rm_dir = False  # keep artifacts under out_dir/exp_name
checkpoints = True
save_every_epoch = True  # save an epoch-level model/optimizer/scaler bundle
rm_intermediaries = False  # retain every completed epoch by default
resume = True

n_cpu = 16
gpu = "0"

In [ ]:
with open("./gen_parameters.json", "r") as file:
    params = json.load(file)

# Copy the selected dictionaries so this notebook never edits gen_parameters.json.
train_params = dict(params[dimensionality]["train"])
test_params = dict(params[dimensionality]["test"])
train_params.update(
    {
        "dimensionality": dimensionality,
        "rm_dir": rm_dir,
        "out_dir": out_dir,
        "n_cpu": n_cpu,
        "cuda_device": gpu,
        "in_path": in_path,
        "train_gen": True,
        "exp_name": exp_name,
        "checkpoints": checkpoints,
        "save_every_epoch": save_every_epoch,
        "resume": resume,
        "rm_intermediaries": rm_intermediaries,
    }
)

train_cd = TrainingClass(train_params)
train_cd.vars_dict()
validation_tracker = ValidationTracker(
    test_dir=test_dir,
    test_params=test_params,
    output_dir=validation_output_dir,
)
train_cd.epoch_end_callback = validation_tracker
print(f"Found {len(validation_tracker.validation_paths)} validation TIFF(s).")

## Train and validate

In [ ]:
train_cd.run()
gc.collect()
torch.cuda.empty_cache()

Each row below pairs one epoch's training and validation losses. The same history is appended to `<exp_name>_train_validation_losses.csv`; per-file values go to `<exp_name>_validation_files.csv`. Validation TIFFs keep stable, source-relative output paths below the validation output directory, so a later epoch overwrites the preceding denoised file instead of creating another copy.

In [ ]:
validation_tracker.history